# Online Experiment Statistics: z-test, Bootstrap, Power, SRM and CUPED — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

Your new ranking model beats the old one by +1.2 % AUC offline. The interviewer's next question is
always: ***how do you know it is better online?*** Offline metrics (`v4/ml-fundamentals/classification-metrics`)
are computed on logs collected by the *old* model and on proxy labels. The ground truth is a randomized
controlled experiment: users are split at random between control **A** (old model) and treatment **B**
(new model), and a small set of statistics decides whether the difference is real. An Applied-MLE
candidate is expected to write those statistics and to explain them:

| question | tool |
|---|---|
| Did click-through change? | two-proportion **z-test** and a **confidence interval** for $p_B - p_A$ |
| Did revenue / watch time per user change? (heavy-tailed, no closed form you trust) | **bootstrap** CI |
| How many users do we need *before* we start? | **power analysis** → sample size per arm |
| Can we trust the split at all? | **sample-ratio-mismatch (SRM)** test |
| Can we detect smaller effects with the same traffic? | **CUPED** variance reduction |

The formulas, with $\Phi$ the standard normal CDF, $z_q = \Phi^{-1}(q)$, $\hat p_A = x_A/n_A$ and
$\hat p_B = x_B/n_B$ (conversions over units):

$$z = \frac{\hat p_B - \hat p_A}{\sqrt{\hat p\,(1-\hat p)\,\big(\tfrac{1}{n_A}+\tfrac{1}{n_B}\big)}},\quad
\hat p = \frac{x_A + x_B}{n_A + n_B},\qquad p\text{-value} = 2\big(1 - \Phi(|z|)\big)$$

$$\text{CI}_{1-\alpha} = \hat p_B - \hat p_A \;\pm\; z_{1-\alpha/2}\sqrt{\tfrac{\hat p_A(1-\hat p_A)}{n_A} + \tfrac{\hat p_B(1-\hat p_B)}{n_B}}$$

$$n_{\text{per arm}} = \left\lceil \frac{\Big(z_{1-\alpha/2}\sqrt{2\bar p(1-\bar p)} + z_{\text{power}}\sqrt{p_1(1-p_1) + p_2(1-p_2)}\Big)^2}{(p_2 - p_1)^2} \right\rceil,
\quad p_1 = p_{\text{base}},\; p_2 = p_{\text{base}} + \text{MDE},\; \bar p = \tfrac{p_1 + p_2}{2}$$

The sample-size formula is the standard normal approximation without continuity correction (Fleiss,
Levin & Paik, *Statistical Methods for Rates and Proportions*). It is what R's `power.prop.test` solves
for and what statsmodels' `samplesize_proportions_2indep_onetail` returns. Lehr's rule of thumb
$n \approx 16\sigma^2/\delta^2$ in `study/ml-system-design.md` (Phase 9) is its simplification for
$\alpha = 0.05$ and 80 % power: for a 5 % CTR and $\delta = 0.001$ this formula gives 752,703 users per
arm, against Lehr's ≈ 760K.

CUPED (Deng, Xu, Kohavi & Walker, WSDM 2013) uses each user's **pre-experiment** value $x$ of the
metric. The pre-experiment value is independent of the assignment, so it can explain variance without
introducing bias:

$$\theta = \frac{\operatorname{cov}(y, x)}{\operatorname{var}(x)}, \qquad y^{\text{cuped}} = y - \theta\,(x - \bar x),
\qquad \operatorname{var}(y^{\text{cuped}}) = \operatorname{var}(y)\,(1 - \rho_{xy}^2).$$

---

### Requirements

Arm **A** is control and arm **B** is treatment. Every difference is **B − A**. Compute in
**float64**: `torch.as_tensor(x, dtype=torch.float64)`. Dividing int tensors would give float32.

1. **`two_proportion_ztest(x_a, n_a, x_b, n_b) -> (diff, z, p_value)`**
   - `diff` $= \hat p_B - \hat p_A$, `z` uses the **pooled** standard error above (the convention of
     statsmodels' `proportions_ztest`), and `p_value` is **two-sided**.
   - If the pooled SE is 0 (no conversions at all, or all units converted), return `z = 0`, `p = 1`
     rather than NaN.
   - Arguments are Python numbers or integer/float tensors that broadcast **elementwise**, so that one
     call can analyse many experiments. Return float64 tensors (0-dim for scalar inputs).
2. **`diff_ci(x_a, n_a, x_b, n_b, alpha=0.05) -> (lo, hi)`**: the Wald interval above with the
   **unpooled** SE and $z_{1-\alpha/2}$. Elementwise like the test.
   *Why two SEs?* The test asks how surprising the data would be **if $p_A = p_B$**, so it estimates
   that common $p$ from both arms. The interval describes where $p_B - p_A$ plausibly lies, so it
   uses each arm's own rate. Near the boundary the two can disagree (Test 1 shows a case).
3. **`bootstrap_ci(a, b, n_boot=10_000, alpha=0.05, generator=None) -> (lo, hi)`**: percentile
   bootstrap CI for $\operatorname{mean}(b) - \operatorname{mean}(a)$.
   - `a` `(n_a,)` and `b` `(n_b,)` are per-unit metric values (e.g. revenue per user). `n_a` may
     differ from `n_b`.
   - Each replicate resamples `n_a` values of `a` **and, independently,** `n_b` values of `b`, with
     replacement, and records $\operatorname{mean}(b^*) - \operatorname{mean}(a^*)$.
   - Return the $\alpha/2$ and $1 - \alpha/2$ quantiles of the `n_boot` replicates (`torch.quantile`'s
     default interpolation is fine) as two scalars.
   - **Vectorized**: no Python loop over replicates. Draw every random number from `generator`, so
     the same seed gives the same interval. Don't modify `a` or `b`.
4. **`sample_size_per_arm(p_base, mde_abs, alpha=0.05, power=0.8) -> int`**: **exactly** the formula
   above for a two-sided two-proportion test with equal allocation: the pooled $2\bar p(1-\bar p)$ next
   to $z_{1-\alpha/2}$ and the unpooled $p_1(1-p_1) + p_2(1-p_2)$ next to $z_{\text{power}}$. It is
   graded to the unit, so Lehr's rule or an all-unpooled variant (fine as back-of-envelope answers)
   will not match. `mde_abs` is an *absolute* difference and may be negative. Round **up**.
5. **`srm_pvalue(n_a, n_b, expected_ratio=0.5) -> p_value`**: Pearson chi-square goodness-of-fit
   test (1 degree of freedom, no continuity correction) of the observed counts against the designed
   split. `expected_ratio` is the expected **share of arm A** (0.5 for 50/50, 0.9 for a 90/10 split).
   With $N = n_A + n_B$:
   $\chi^2 = \frac{(n_A - rN)^2}{rN} + \frac{(n_B - (1-r)N)^2}{(1-r)N}$ and
   $p = P(\chi^2_1 > \chi^2)$. Elementwise on tensors.
6. **`cuped(y, x_pre) -> (y_adj, theta)`**: `y`, `x_pre` are `(n,)` tensors (in-experiment metric,
   the same metric in the pre-period). Compute $\theta$ with the **same** normalization in the
   covariance and the variance, and $y_{\text{adj}} = y - \theta(x - \bar x)$. Use $\theta = 0$ if
   $\operatorname{var}(x) = 0$. Return `y_adj` `(n,)` and `theta` as a scalar. In an analysis you call
   it **once on the pooled data of both arms** (one $\theta$) and then compare the arm means of
   `y_adj`.

---

### Constraints

- ✅ `torch.special.ndtr` ($\Phi$), `torch.special.ndtri` ($\Phi^{-1}$), `torch.special.erfc`,
  `torch.quantile`, `torch.randint(..., generator=generator)`, `math`.
- ❌ No `scipy`, `statsmodels` or `numpy` inside your functions.
- ❌ No Python loop over bootstrap replicates. The grader counts torch calls for small and large
  `n_boot`.
- ❌ Don't make $\theta$'s covariance and variance use different `ddof` (e.g. `torch.cov` (ddof 1)
  over `x.var(unbiased=False)`).

---

<details>
  <summary>💡 Hint</summary>

  - Under $H_0$ both arms share one conversion rate. What is your best estimate of it?
  - $\Phi(-|z|)$ is one tail. How many tails does a two-sided test count? What is the distribution of
    $z^2$ when $z \sim N(0, 1)$, and what does that say about a 1-dof chi-square tail?
  - For `n_boot` replicates of an arm with `n` values you need an index matrix of shape `(n_boot, n)`.
    What should the second arm's index matrix look like, and why must it be a *separate* draw?
  - Sample size: the test rejects when $|\hat\delta| > z_{1-\alpha/2}\,\mathrm{SE}_0$. If the true
    difference is $\delta$, $\hat\delta \approx N(\delta, \mathrm{SE}_1^2)$. When is the rejection
    probability equal to `power`? (Ignore the far tail.)
  - CUPED's $\theta$ is a regression slope. Why does subtracting $\theta(x - \bar x)$ leave the mean,
    and therefore the treatment effect, unchanged?

</details>

In [ ]:
import math

import torch

F64 = torch.float64

In [ ]:
g = torch.Generator().manual_seed(0)

# --- Experiment 1: click-through rate, users randomized 50/50 between the old (A) and new (B) model.
n_a, n_b = 200_000, 200_400
clicks_a = int((torch.rand(n_a, generator=g) < 0.0500).sum())
clicks_b = int((torch.rand(n_b, generator=g) < 0.0520).sum())       # true lift +0.2 points
print(f"CTR  A: {clicks_a}/{n_a} = {clicks_a / n_a:.4%}   B: {clicks_b}/{n_b} = {clicks_b / n_b:.4%}")

# --- Experiment 2: watch time per user (heavy-tailed), with each user's PRE-experiment watch time.
m = 3000                                                            # users per arm
taste = torch.randn(2 * m, generator=g, dtype=F64)                  # stable per-user propensity
x_pre = torch.exp(1.0 + 0.8 * taste + 0.3 * torch.randn(2 * m, generator=g, dtype=F64))
y_post = torch.exp(1.0 + 0.8 * taste + 0.3 * torch.randn(2 * m, generator=g, dtype=F64))
arm = torch.cat([torch.zeros(m, dtype=torch.long), torch.ones(m, dtype=torch.long)])
y_post[arm == 1] *= 1.05                                            # true lift +5 %
watch_a, watch_b = y_post[arm == 0], y_post[arm == 1]
print(f"watch time  A: mean {watch_a.mean():.3f}  B: mean {watch_b.mean():.3f}  "
      f"(max {y_post.max():.1f}: heavy right tail)")

In [ ]:
def _f64(v):
    return torch.as_tensor(v, dtype=F64)        # ints -> float64; int / int would give float32


def two_proportion_ztest(x_a, n_a, x_b, n_b):
    """Two-sided two-proportion z-test with the pooled standard error.

    Args:
        x_a, n_a: conversions and units in control (numbers or tensors, broadcast elementwise).
        x_b, n_b: conversions and units in treatment.

    Returns:
        diff: p_B - p_A, z: diff / pooled SE (0 if the SE is 0), p_value: 2 * (1 - Phi(|z|)).
        float64 tensors of the broadcast shape.
    """
    x_a, n_a, x_b, n_b = map(_f64, (x_a, n_a, x_b, n_b))
    diff = x_b / n_b - x_a / n_a
    p_pool = (x_a + x_b) / (n_a + n_b)                      # H0: one shared rate
    se0 = torch.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    ok = se0 > 0
    z = torch.where(ok, diff / torch.where(ok, se0, torch.ones_like(se0)), torch.zeros_like(diff))
    p_value = torch.special.erfc(z.abs() / math.sqrt(2.0))  # = 2 * (1 - Phi(|z|)), accurate in the tail
    return diff, z, p_value

In [ ]:
def diff_ci(x_a, n_a, x_b, n_b, alpha=0.05):
    """Wald (1 - alpha) confidence interval for p_B - p_A with the UNPOOLED standard error.

    Returns:
        (lo, hi): float64 tensors of the broadcast shape.
    """
    x_a, n_a, x_b, n_b = map(_f64, (x_a, n_a, x_b, n_b))
    p_a, p_b = x_a / n_a, x_b / n_b
    se1 = torch.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    z = torch.special.ndtri(_f64(1 - alpha / 2))            # 1.96 for alpha = 0.05
    diff = p_b - p_a
    return diff - z * se1, diff + z * se1

In [ ]:
def bootstrap_ci(a, b, n_boot=10_000, alpha=0.05, generator=None):
    """Percentile bootstrap CI for mean(b) - mean(a), resampling each arm independently.

    Args:
        a: (n_a,) per-unit metric in control.  b: (n_b,) per-unit metric in treatment.
        n_boot: number of bootstrap replicates.  alpha: 1 - confidence level.
        generator: torch.Generator for every random draw (None = global RNG).

    Returns:
        (lo, hi): the alpha/2 and 1 - alpha/2 quantiles of the replicate differences.
    """
    a, b = _f64(a), _f64(b)
    # One row per replicate; separate draws for the two arms (they are independent samples).
    idx_a = torch.randint(a.numel(), (n_boot, a.numel()), generator=generator)
    idx_b = torch.randint(b.numel(), (n_boot, b.numel()), generator=generator)
    diffs = b[idx_b].mean(dim=1) - a[idx_a].mean(dim=1)     # (n_boot,)
    q = torch.tensor([alpha / 2, 1 - alpha / 2], dtype=F64)
    lo, hi = torch.quantile(diffs, q)
    return lo, hi

In [ ]:
def sample_size_per_arm(p_base, mde_abs, alpha=0.05, power=0.8):
    """Units per arm for a two-sided two-proportion z-test (pooled SE under H0, unpooled under H1).

    Returns:
        int, rounded up.
    """
    p1 = float(p_base)
    p2 = p1 + float(mde_abs)
    p_bar = (p1 + p2) / 2
    z_alpha = torch.special.ndtri(_f64(1 - alpha / 2)).item()   # two-sided: alpha / 2 per tail
    z_power = torch.special.ndtri(_f64(power)).item()
    sd_h0 = math.sqrt(2 * p_bar * (1 - p_bar))                   # sqrt(n) * SE under H0
    sd_h1 = math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))             # sqrt(n) * SE under H1
    n = ((z_alpha * sd_h0 + z_power * sd_h1) / (p2 - p1)) ** 2
    return math.ceil(n)

In [ ]:
def srm_pvalue(n_a, n_b, expected_ratio=0.5):
    """Sample-ratio-mismatch test: Pearson chi-square (1 dof) of the observed split vs the design.

    Args:
        n_a, n_b: units observed in A and B (numbers or tensors).
        expected_ratio: designed share of arm A.

    Returns:
        p-value (float64 tensor).
    """
    n_a, n_b = _f64(n_a), _f64(n_b)
    total = n_a + n_b
    e_a, e_b = total * expected_ratio, total * (1 - expected_ratio)
    chi2 = (n_a - e_a) ** 2 / e_a + (n_b - e_b) ** 2 / e_b
    return torch.special.erfc(torch.sqrt(chi2 / 2))   # P(Z^2 > chi2) = 2 * (1 - Phi(sqrt(chi2)))

In [ ]:
def cuped(y, x_pre):
    """CUPED adjustment with the pre-experiment covariate x_pre.

    Args:
        y: (n,) in-experiment metric.  x_pre: (n,) the same metric before the experiment.

    Returns:
        y_adj: (n,) y - theta * (x_pre - mean(x_pre)),  theta: cov(y, x) / var(x) (0 if var(x) == 0).
    """
    y, x = _f64(y), _f64(x_pre)
    xc = x - x.mean()
    var_x = (xc * xc).mean()                          # 1/n in both: the normalization cancels
    cov_xy = (xc * (y - y.mean())).mean()
    theta = cov_xy / var_x if var_x > 0 else torch.zeros((), dtype=F64)
    return y - theta * xc, theta

In [ ]:
# Test 1: z-test and Wald CI on the CTR experiment, against the formulas written out by hand
diff, z, p = two_proportion_ztest(clicks_a, n_a, clicks_b, n_b)
lo, hi = diff_ci(clicks_a, n_a, clicks_b, n_b)
pa, pb = clicks_a / n_a, clicks_b / n_b
pp = (clicks_a + clicks_b) / (n_a + n_b)
z_hand = (pb - pa) / math.sqrt(pp * (1 - pp) * (1 / n_a + 1 / n_b))
half = 1.959963984540054 * math.sqrt(pa * (1 - pa) / n_a + pb * (1 - pb) / n_b)
assert torch.is_tensor(z) and z.dtype == F64
assert abs(float(diff) - (pb - pa)) < 1e-12 and abs(float(z) - z_hand) < 1e-9
assert abs(float(p) - math.erfc(abs(z_hand) / math.sqrt(2))) < 1e-12
assert abs(float(lo) - (pb - pa - half)) < 1e-12 and abs(float(hi) - (pb - pa + half)) < 1e-12
print(f"CTR lift {float(diff):+.4%}, z = {float(z):.2f}, p = {float(p):.2e}, "
      f"95% CI [{float(lo):+.4%}, {float(hi):+.4%}]")

# symmetry, edge case, vectorization
d2, z2, p2 = two_proportion_ztest(clicks_b, n_b, clicks_a, n_a)
assert torch.allclose(d2, -diff) and torch.allclose(z2, -z) and torch.allclose(p2, p)
_, z0, p0 = two_proportion_ztest(0, 500, 0, 700)
assert float(z0) == 0 and float(p0) == 1, "pooled SE = 0 must give z = 0, p = 1 (not NaN)"
dv, zv, pv = two_proportion_ztest(torch.tensor([30, 500]), torch.tensor([400, 10_000]),
                                  torch.tensor([61, 560]), torch.tensor([1300, 10_000]))
assert zv.shape == (2,) and abs(float(zv[1]) - float(two_proportion_ztest(500, 10_000, 560, 10_000)[1])) < 1e-12

# Pooled test vs unpooled CI can disagree near the boundary:
_, z_e, p_e = two_proportion_ztest(30, 400, 61, 1300)
lo_e, hi_e = diff_ci(30, 400, 61, 1300)
print(f"30/400 vs 61/1300: test p = {float(p_e):.4f} (< 0.05) but the 95% CI "
      f"[{float(lo_e):+.4f}, {float(hi_e):+.4f}] contains 0")
assert float(p_e) < 0.05 and float(lo_e) < 0 < float(hi_e)
print("Test 1 passed")

In [ ]:
# Test 2: 20,000 A/A tests in ONE vectorized call: the false-positive rate must be ~alpha
gen = torch.Generator().manual_seed(1)
n_exp, n_users, ctr = 20_000, 10_000, 0.05
x_a_sim = torch.binomial(torch.full((n_exp,), float(n_users)), torch.full((n_exp,), ctr), generator=gen)
x_b_sim = torch.binomial(torch.full((n_exp,), float(n_users)), torch.full((n_exp,), ctr), generator=gen)
_, _, p_aa = two_proportion_ztest(x_a_sim, n_users, x_b_sim, n_users)
fpr = (p_aa < 0.05).double().mean().item()
print(f"A/A false-positive rate at alpha = 0.05: {fpr:.4f}   P(p < 0.5) = {(p_aa < 0.5).double().mean():.3f}")
assert p_aa.shape == (n_exp,) and abs(fpr - 0.05) < 0.006, "under H0, p < 0.05 should happen ~5% of the time"
print("Test 2 passed")

In [ ]:
# Test 3: bootstrap CI for the watch-time lift
est = (watch_b.mean() - watch_a.mean()).item()
ci1 = bootstrap_ci(watch_a, watch_b, n_boot=2000, generator=torch.Generator().manual_seed(7))
ci2 = bootstrap_ci(watch_a, watch_b, n_boot=2000, generator=torch.Generator().manual_seed(7))
lo, hi = float(ci1[0]), float(ci1[1])
assert (lo, hi) == (float(ci2[0]), float(ci2[1])), "same generator seed -> same interval"
assert lo < est < hi
se = math.sqrt(watch_a.var(unbiased=False) / m + watch_b.var(unbiased=False) / m)
print(f"watch-time lift {est:+.3f}; bootstrap 95% CI [{lo:+.3f}, {hi:+.3f}]; "
      f"normal-theory CI [{est - 1.96 * se:+.3f}, {est + 1.96 * se:+.3f}]")
assert abs((hi - lo) / (2 * 1.96 * se) - 1) < 0.1, "with 3000 users per arm both CIs should be similar"

# On a small, right-skewed sample the percentile interval is asymmetric (longer upper arm)
small_b = watch_b[:25]
lo_s, hi_s = bootstrap_ci(torch.zeros(25, dtype=F64), small_b, n_boot=4000,
                          generator=torch.Generator().manual_seed(8))
est_s = small_b.mean().item()
print(f"25 skewed values: mean {est_s:.2f}, CI [{float(lo_s):.2f}, {float(hi_s):.2f}]")
assert float(hi_s) - est_s > est_s - float(lo_s)
print("Test 3 passed")

In [ ]:
# Test 4: sample size — the formula, Lehr's rule of thumb, and the power it actually delivers
n_ctr = sample_size_per_arm(0.05, 0.001)
lehr = 16 * 0.05 * 0.95 / 0.001 ** 2
print(f"5% CTR, MDE +0.1 point: {n_ctr:,} users per arm (Lehr's 16*s^2/d^2: {lehr:,.0f})")
assert n_ctr == 752_703 and isinstance(n_ctr, int)
assert sample_size_per_arm(0.10, 0.02) == 3841 and sample_size_per_arm(0.30, -0.05) == 1251
assert sample_size_per_arm(0.1, 0.01) > sample_size_per_arm(0.1, 0.02)              # smaller MDE
assert sample_size_per_arm(0.1, 0.02, power=0.9) > sample_size_per_arm(0.1, 0.02)   # more power

# Simulate 4000 experiments of that size with a true +2-point lift: ~80% should reach p < 0.05
n_req = sample_size_per_arm(0.10, 0.02)
gen = torch.Generator().manual_seed(2)
xa = torch.binomial(torch.full((4000,), float(n_req)), torch.full((4000,), 0.10), generator=gen)
xb = torch.binomial(torch.full((4000,), float(n_req)), torch.full((4000,), 0.12), generator=gen)
achieved = (two_proportion_ztest(xa, n_req, xb, n_req)[2] < 0.05).double().mean().item()
print(f"n = {n_req} per arm -> simulated power {achieved:.3f} (target 0.80)")
assert abs(achieved - 0.80) < 0.03
print("Test 4 passed")

In [ ]:
# Test 5: sample-ratio mismatch
p_ok = float(srm_pvalue(n_a, n_b))
p_bug = float(srm_pvalue(200_000, 197_000))           # e.g. a bot filter that drops 1.5% of treatment
p_9010 = float(srm_pvalue(9_000, 1_100, expected_ratio=0.9))
print(f"our split {n_a} vs {n_b}: p = {p_ok:.3f};  200,000 vs 197,000: p = {p_bug:.1e};  "
      f"90/10 design with 9,000 vs 1,100: p = {p_9010:.4f}")
chi2 = (50_000 - 50_500) ** 2 / 50_500 + (51_000 - 50_500) ** 2 / 50_500
assert abs(float(srm_pvalue(50_000, 51_000)) - math.erfc(math.sqrt(chi2 / 2))) < 1e-15
assert p_ok > 0.05 and p_bug < 1e-3 and float(srm_pvalue(1000, 1000)) == 1.0
assert abs(p_9010 - 0.0028347) < 1e-6
print("Test 5 passed")

In [ ]:
# Test 6: CUPED on the watch-time experiment (theta from the pooled data of both arms)
y_adj, theta = cuped(y_post, x_pre)
rho = torch.corrcoef(torch.stack([y_post, x_pre]))[0, 1].item()
assert abs(y_adj.mean().item() - y_post.mean().item()) < 1e-9, "CUPED must not change the mean"
ratio = (y_adj.var() / y_post.var()).item()
assert abs(ratio - (1 - rho ** 2)) < 1e-9, "the optimal theta reduces the variance by exactly (1 - rho^2)"
xc = x_pre - x_pre.mean()
assert abs(float(theta) - ((xc * (y_post - y_post.mean())).mean() / (xc * xc).mean()).item()) < 1e-9

def welch(a, b):
    est = (b.mean() - a.mean()).item()
    se = math.sqrt(a.var().item() / a.numel() + b.var().item() / b.numel())
    return est, se, math.erfc(abs(est / se) / math.sqrt(2))

raw = welch(watch_a, watch_b)
adj = welch(y_adj[arm == 0], y_adj[arm == 1])
print(f"theta = {float(theta):.3f}, rho = {rho:.3f}: variance x {ratio:.3f}")
print(f"lift without CUPED {raw[0]:+.3f} +/- {1.96 * raw[1]:.3f} (p = {raw[2]:.3f});  "
      f"with CUPED {adj[0]:+.3f} +/- {1.96 * adj[1]:.3f} (p = {adj[2]:.4f})")
print(f"(true lift: 5% of the mean watch time = {0.05 * watch_a.mean().item():.3f} approximately; the raw "
      f"estimate also carries a chance imbalance in pre-period watch time that CUPED removes)")
assert adj[1] < raw[1]
y_c, th_c = cuped(y_post, torch.full_like(x_pre, 2.0))
assert float(th_c) == 0 and torch.equal(y_c, y_post), "a constant covariate gives theta = 0"
print("Test 6 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("ab-testing", two_proportion_ztest=two_proportion_ztest, diff_ci=diff_ci,
      bootstrap_ci=bootstrap_ci, sample_size_per_arm=sample_size_per_arm,
      srm_pvalue=srm_pvalue, cuped=cuped)

### Demo: two classic ways to fool yourself

**Peeking.** Run an A/A test and check the p-value after every 10 % of the traffic, stopping at the
first $p < 0.05$. Every individual look is a valid 5 % test, but taking the minimum over ten looks is
not. The z-test is vectorized, so 4,000 experiments × 10 looks is a single `(4000, 10)` call.

**A post-treatment covariate in CUPED.** CUPED is unbiased because $x$ is measured *before*
assignment, so $E[\bar x_B - \bar x_A] = 0$. If you use an in-experiment metric that the treatment
also moves (here: sessions during the test), you subtract part of the treatment effect.

In [ ]:
# Peeking: 4000 A/A experiments, 10 interim looks each
gen = torch.Generator().manual_seed(3)
n_exp, looks, per_look, ctr = 4000, 10, 2000, 0.05
inc_a = torch.binomial(torch.full((n_exp, looks), float(per_look)), torch.full((n_exp, looks), ctr), generator=gen)
inc_b = torch.binomial(torch.full((n_exp, looks), float(per_look)), torch.full((n_exp, looks), ctr), generator=gen)
n_seen = per_look * torch.arange(1, looks + 1, dtype=F64)                      # (10,) broadcasts
_, _, p_looks = two_proportion_ztest(inc_a.cumsum(1), n_seen, inc_b.cumsum(1), n_seen)
print(f"false-positive rate, one look at the end: {(p_looks[:, -1] < 0.05).double().mean():.3f}")
print(f"false-positive rate, stop at the first of 10 looks with p < 0.05: "
      f"{(p_looks < 0.05).any(dim=1).double().mean():.3f}")

# CUPED with a covariate that the treatment moves -> biased effect
gen = torch.Generator().manual_seed(4)
n = 20_000
assign = torch.randint(0, 2, (n,), generator=gen).bool()
engagement = torch.randn(n, generator=gen, dtype=F64)
sessions = 5 + engagement + 0.5 * assign + 0.3 * torch.randn(n, generator=gen, dtype=F64)   # treated -> more sessions
y = 10 + 2 * engagement + 1.0 * assign + torch.randn(n, generator=gen, dtype=F64)           # true effect = 1.0
pre = 5 + engagement + 0.3 * torch.randn(n, generator=gen, dtype=F64)                      # pre-period sessions
for name, cov in (("pre-period sessions", pre), ("in-experiment sessions", sessions)):
    adj, th = cuped(y, cov)
    print(f"CUPED with {name:24s}: effect {(adj[assign].mean() - adj[~assign].mean()).item():.3f} (true 1.0)")

## Why it works

**z-test.** By the CLT, $\hat p_B - \hat p_A$ is approximately normal with variance
$\frac{p_A(1-p_A)}{n_A} + \frac{p_B(1-p_B)}{n_B}$. Under $H_0$, $p_A = p_B = p$, so the variance is
$p(1-p)(\frac1{n_A} + \frac1{n_B})$, and the maximum-likelihood estimate of the common $p$ under $H_0$
pools both arms.
$|z| > z_{1-\alpha/2}$ then rejects with probability $\alpha$ under $H_0$ (Test 2 measures about 5 %).
The two-sided p-value is $2\Phi(-|z|) = \operatorname{erfc}(|z|/\sqrt2)$. The `erfc` form keeps its
relative precision far into the tail, where `1 - ndtr(|z|)` rounds to 0. The test is algebraically
the Pearson chi-square test of the 2×2 table: $z^2 = \chi^2_1$.

**Wald CI.** It inverts the Wald test, whose standard error is evaluated at the unrestricted
estimates $\hat p_A, \hat p_B$ (nobody assumes $p_A = p_B$), so each arm keeps its own variance. Because the two SEs differ, the CI and the test can
disagree in borderline cases (Test 1). Report the CI for the effect size and the pooled test for the
decision, and say which one you used.

**Bootstrap.** It is the plug-in principle: treat each arm's empirical distribution as its population
and resample to see how $\bar b - \bar a$ would vary. The arms are independent samples, so each is
resampled from its own data with its own $n$. Resampling the pooled data would simulate $H_0$, which is
a permutation-style *test*, not an interval for the effect. The bootstrap standard deviation is
$\approx \sqrt{\hat\sigma_a^2/n_a + \hat\sigma_b^2/n_b}$. The **percentile** interval takes the
replicate quantiles directly, so it follows skew in the sampling distribution (Test 3's small sample).
It is transformation-respecting but can under-cover for small $n$. BCa corrects for both bias and skew.

**Sample size.** The test rejects (in the direction of the effect) when
$\hat\delta > z_{1-\alpha/2}\,\sigma_0/\sqrt n$, where $\sigma_0^2 = 2\bar p(1-\bar p)$ is the per-arm variance factor under $H_0$.
Under $H_1$, $\hat\delta \sim N(\delta, \sigma_1^2/n)$ with $\sigma_1^2 = p_1q_1 + p_2q_2$. Setting
$P(\text{reject}) = \Phi\big((\sqrt n|\delta| - z_{1-\alpha/2}\sigma_0)/\sigma_1\big)$ equal to the
target power gives $\sqrt n\,|\delta| = z_{1-\alpha/2}\sigma_0 + z_{\text{power}}\sigma_1$. That is the
formula; the opposite rejection tail is ignored, as in R's `power.prop.test` (default `strict = FALSE`).
With $\sigma_0 \approx \sigma_1 \approx \sqrt2\,\sigma$ and $(1.96 + 0.84)^2 \cdot 2 \approx 15.7$ you get
Lehr's $16\sigma^2/\delta^2$. Test 4 checks by simulation that the formula delivers ≈ 80 % power.

**SRM.** With $N$ units randomized to A with probability $r$, $n_A \sim \text{Binomial}(N, r)$, and
$\chi^2 = (n_A - rN)^2/(N r(1-r))$ is the square of a one-proportion z-statistic, so
$P(\chi^2_1 > s) = 2\Phi(-\sqrt s) = \operatorname{erfc}(\sqrt{s/2})$.

**CUPED.** For any constant $\theta$, $E[y - \theta(x - \bar x)]$ has the same mean, and because $x$ is
pre-experiment, $E[\bar x_B - \bar x_A] = 0$, so the treatment effect estimate stays unbiased.
$\operatorname{var}(y - \theta x) = \operatorname{var}(y) - 2\theta\operatorname{cov}(x,y) + \theta^2\operatorname{var}(x)$
is minimized at $\theta = \operatorname{cov}/\operatorname{var}(x)$ (the OLS slope), with minimum
$\operatorname{var}(y)(1-\rho^2)$. This holds exactly in-sample when the covariance and the variance
share one normalization. A correlation of 0.7 halves the variance, which halves the users (or the
time) needed for the same power. It is essentially regression adjustment (ANCOVA) with the pre-period
metric as the covariate.

## Complexity

The z-test, CI, sample size, SRM and CUPED are $O(1)$ per experiment ($O(n)$ for CUPED's moments)
and vectorize over thousands of experiments. The bootstrap is $O(n_{\text{boot}}\,(n_a + n_b))$ time
and, as written, the same memory for the index matrices. For millions of users, process chunks of
replicates, aggregate per-user values into buckets first, or use the **Poisson bootstrap** (weights
$\sim \text{Poisson}(1)$ per unit, computable in one streaming pass; Chamandy et al., Google, 2012).

## Common bugs

- Unpooled SE in the test or pooled SE in the CI, or not saying which one you used.
- A one-sided p-value ($1 - \Phi(z)$ or $\Phi(-|z|)$) reported as two-sided.
- `int_tensor / int_tensor` silently computed in float32. `1 - ndtr(|z|)` loses the tail (p = 0 for
  $|z| > 8.3$).
- NaN from $0/0$ when neither arm converted.
- Bootstrap: resampling the pooled data (centres the interval on 0), resampling *pairs* with one index
  for both arms (wrong for unpaired arms, and impossible when $n_a \ne n_b$), sampling without
  replacement (zero width), quantiles $\alpha$ / $1-\alpha$ instead of $\alpha/2$ / $1-\alpha/2$, a
  Python loop over 10,000 replicates, or ignoring the generator so results are not reproducible.
- Sample size with $z_{1-\alpha}$ instead of $z_{1-\alpha/2}$, a relative MDE plugged in as an absolute
  one, rounding down, or reporting the total as the per-arm size.
- SRM with `expected_ratio` read as B's share, or a continuity correction the design did not ask for.
- CUPED: dividing by $\operatorname{var}(y)$; `torch.cov` (ddof 1) over `x.var(unbiased=False)`;
  forgetting to centre $x$ (shifts the mean); using an in-experiment covariate (biased, demo above).
  The clean, standard choice is one $\theta$ fitted on the pooled data of both arms.

## Interview follow-ups

1. **"We checked the dashboard every day and stopped when it went significant. Is that OK?"** No.
   With repeated looks the chance that *some* look crosses $p < 0.05$ is far above 5 % (the demo
   measures it for 10 looks). Either fix the horizon in advance (from the power analysis), or use a
   sequential design that spends $\alpha$ across looks: group-sequential O'Brien–Fleming / Pocock
   boundaries via Lan–DeMets spending functions, or always-valid p-values / mSPRT
   (Johari et al., *Always Valid Inference*).
2. **"We track 20 metrics; three moved at p < 0.05."** That is about what chance alone produces
   ($20 \times 0.05 = 1$ expected false positive, and correlated metrics make it lumpier). Pre-register
   one primary metric (the OEC) plus guardrails. For families of metrics control the FWER with
   Bonferroni ($\alpha/m$) or the uniformly more powerful Holm step-down, or the FDR with
   Benjamini–Hochberg: reject the $k$ smallest p-values, where $k$ is the largest index with
   $p_{(k)} \le k\alpha/m$.
3. **Novelty, primacy and long-term effects.** Users react to change itself: novelty inflates early
   engagement and primacy depresses it. Run at least one or two full weekly cycles, plot the effect
   by day since first exposure (or analyse new users separately), and keep long-term holdouts for
   effects that build slowly (ads blindness, retention).
4. **Interference and network effects.** The analysis assumes one user's outcome does not depend on
   other users' assignment (SUTVA). In marketplaces (shared supply, budgets, ranking slots) and social
   products (messages, feeds) it does, and the naive estimate is biased. Use cluster randomization
   (graph clusters, ego networks; Ugander et al., 2013), geo experiments, switchback (time-sliced)
   designs, or budget-split designs for ads, and analyse at the cluster level.
5. **Ratio metrics and the delta method.** CTR = clicks / page views, randomized by *user*: page
   views within a user are correlated, so treating them as i.i.d. Bernoulli trials understates the
   variance. Treat it as $\bar X / \bar Y$ over users and use the delta method,
   $\operatorname{var}(\bar X/\bar Y) \approx \frac{1}{\mu_Y^2}\big(\operatorname{var}\bar X - 2\frac{\mu_X}{\mu_Y}\operatorname{cov}(\bar X,\bar Y) + \frac{\mu_X^2}{\mu_Y^2}\operatorname{var}\bar Y\big)$
   (Deng, Knoblich & Lu, KDD 2018), or bootstrap over users. The analysis unit must match the
   randomization unit.
6. **Why does an SRM invalidate the experiment instead of being a nuisance?** Randomization
   guarantees comparable arms only if *every* assigned unit is counted. A significant SRM means some
   units went missing differently in the two arms: a redirect that loses slow clients, a crash, bot
   filtering, logging that depends on the treatment, a trigger condition the treatment changes. The
   surviving populations differ in unknown ways, so any metric difference may be selection, not
   treatment, and reweighting cannot fix an unknown mechanism. Find the cause, fix it, rerun.
   Platforms typically alarm only at a strict threshold ($p < 0.001$ or lower) because the check runs
   on every experiment.
7. **Offline gains that do not show up online (and vice versa).** Offline metrics are computed on
   logs produced by the old model (exposure and position bias: the new model's top items may never
   have been shown), on proxy labels (clicks, not satisfaction), and ignore latency, feedback loops
   and the fact that users adapt. Off-policy estimators (IPS with logged propensities) narrow the gap.
   Always gate on online results and track the offline-to-online correlation over past launches.
8. **Why interleaving for rankers?** Instead of showing A to some users and B to others, merge both
   rankings into one list per query (team-draft interleaving; Radlinski, Kurup & Joachims, CIKM 2008)
   and credit each click to the ranker that contributed the item. Every user compares both models, so
   between-user variance cancels and a winner emerges with far less traffic ("far more sensitive for
   ranking changes", `study/ml-system-design.md`). It
   measures relative preference, not business metrics like revenue, so it is used to screen candidates
   before a confirmatory A/B test.